# Day 4.5 — Supervisor Synthesis

## Before you begin

### Learning outcomes

- Merge findings from four branches with a rule that does not rely on shared ids.
- Make the supervisor report what it merged and what it truncated.
- Break the merge rule on purpose and watch a real defect disappear.

Architecture reference: [Day 4 diagrams D14](../diagrams/source/day_04.md)

### Expected observation

12 findings arrive, id-based merging leaves 3 duplicates, location-based merging removes them — and a wider merge window silently deletes a genuine defect.


## Concept briefing

## Deduplication is harder than matching IDs

If every reviewer reuses the same seeded id, deduplication is a dictionary lookup. Real
reviewers do not. A static checker calls it `AST-EVAL-20`; a model calls it
`MODEL-SEC-20-3`; a human calls it "unsafe eval". Same defect, three ids, three survivors
in your final report.

So the supervisor merges on a rule that does not depend on shared ids: same category, and
lines close enough to be the same place. That rule is *live*, and it can be wrong. Two
genuinely different security defects on adjacent lines will be **falsely merged** and the
second one is lost. Loosen the rule and you hide real defects; tighten it and duplicates
survive. There is no setting that is right for every artifact, which is why production
systems keep a human in the merge loop.

Evaluation needs the same discipline. Credit each known defect **once**: three reviewers
reporting the same problem is one defect found plus two duplicates, never three finds.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — Collect everything the branches produced

Four groups arrive at the fan-in point: the AST checker plus three specialists. Nobody has merged anything yet.


In [ ]:
from review_team import SPECIALIST_ROLES, deterministic_checks, specialist_review

groups = [deterministic_checks(SOURCE)]
groups += [specialist_review(SOURCE, role) for role in SPECIALIST_ROLES]

labels = ["ast_checker", *SPECIALIST_ROLES]
for label, group in zip(labels, groups):
    print(f"{label:<16} produced {len(group)} finding(s)")

print("\nTotal findings arriving at the supervisor:", sum(len(g) for g in groups))


## Step 3 — The naive rule: merge on identifier

The obvious rule is "two findings are the same if their ids match". Watch what it does when the parser and a specialist describe the same defect under different ids.


In [ ]:
from review_team import synthesize_with_report

by_id = synthesize_with_report(groups, merge_by="id")

print("Received          :", by_id.received)
print("Merged duplicates :", by_id.merged_duplicates)
print("Kept              :", by_id.kept)
print("Dropped over cap  :", by_id.dropped_over_cap)
print()

# Find the pairs that survived: same category, same line, different id.
for i, first in enumerate(by_id.findings):
    for second in by_id.findings[i + 1:]:
        if first.category == second.category and first.line == second.line:
            print(f"SURVIVING DUPLICATE on line {first.line}:")
            print(f"   {first.id:<26} ({first.reviewer})")
            print(f"   {second.id:<26} ({second.reviewer})")


## Step 4 — A rule that does not need shared ids

Real reviewers invent their own ids, so the supervisor merges on **category plus line proximity** instead: same category, lines within one of each other, same defect. Now the overlaps collapse.


In [ ]:
by_location = synthesize_with_report(groups, merge_by="location")

print(f"{'rule':<12}{'received':>10}{'merged':>9}{'kept':>7}")
print("-" * 38)
for report in (by_id, by_location):
    print(f"{report.merge_key:<12}{report.received:>10}{report.merged_duplicates:>9}"
          f"{report.kept:>7}")

print("\nFinal report, ranked by severity then line:")
for finding in by_location.findings:
    print(f"   {finding.severity:<8} line {finding.line:>3}  {finding.title}  "
          f"[{finding.reviewer}]")


## Step 5 — Duplicates that survive are a measurable cost

The evaluator credits each defect once, so surviving duplicates show up in their own column. This is what a bad merge rule costs a reader: the same problem, twice, in a report they have to triage.


In [ ]:
from review_team import ReviewRun, evaluate

for report in (by_id, by_location):
    row = evaluate(ReviewRun(f"merge_by_{report.merge_key}", report.findings), GOLDEN_PATH)
    print(f"merge_by={report.merge_key:<10} report length {row['reported_findings']:>2} | "
          f"defects found {row['found']}/9 | duplicates in report {row['duplicates']}")


## Step 6 — Capping the output must not be silent

A supervisor has to terminate, so it caps the report. If it truncates without saying so, findings vanish and nobody knows. Ours records the number dropped.


In [ ]:
capped = synthesize_with_report(groups, max_findings=4, merge_by="location")

print("Kept             :", capped.kept)
print("Dropped over cap :", capped.dropped_over_cap)
print()
print("Kept (highest severity first):")
for finding in capped.findings:
    print(f"   {finding.severity:<8} line {finding.line:>3}  {finding.title}")

dropped_ids = ({f.id for f in by_location.findings} - {f.id for f in capped.findings})
print("\nDropped, and reported as dropped:", sorted(dropped_ids))


## Step 7 — The merge rule can be wrong

Location-based merging has no way to know whether two nearby findings are one defect or two. Here are two genuinely different security problems one line apart. The rule merges them, and the second one is gone.


In [ ]:
from review_team import Finding, synthesize

first  = Finding("A", "security", 15, "SQL built by string concatenation",
                 "query = ... + customer_name", "critical", "Parameterise it.", "sec_a")
second = Finding("B", "security", 16, "Query result returned without authorisation check",
                 "return database.execute(query)", "high", "Check the caller.", "sec_b")

merged = synthesize([[first, second]], merge_by="location")

print("Two different defects went in:")
for f in (first, second):
    print(f"   line {f.line}  {f.title}")
print("\nCame out of the supervisor:", len(merged), "finding(s)")
for f in merged:
    print(f"   line {f.line}  {f.title}")
print("\nFALSE MERGE: a real defect was deleted by the deduplication rule.")


### Try it yourself

Widen the merge window from 1 line to 5 and predict what happens to the three correctness defects on lines 7, 9 and 25.


In [ ]:
# --- Worked solution ---
correctness = specialist_review(SOURCE, "correctness")
print("Correctness findings before merging:")
for f in correctness:
    print(f"   line {f.line:>3}  {f.id}  {f.title}")

for window in (1, 5):
    report = synthesize_with_report([correctness], merge_by="location", line_window=window)
    row = evaluate(ReviewRun("demo", report.findings), GOLDEN_PATH)
    print(f"\nline_window={window}: kept {report.kept}, merged {report.merged_duplicates}, "
          f"defects credited {row['found']}")
    for f in report.findings:
        print(f"   line {f.line:>3}  {f.title}")

print()
print("With window=5, lines 7 and 9 are treated as one defect and recall falls.")
print("Loosen the rule and you hide real defects; tighten it and duplicates survive.")
print("There is no window that is right for every artifact - which is why production")
print("systems keep a human in the merge loop.")


### Checkpoint

**1. Why can't the supervisor just deduplicate on the finding id?**

<details><summary>Show answer</summary>

Because ids are only shared inside this classroom. A static checker calls the line-20 problem `AST-EVAL-20`, a model calls it `MODEL-SEC-20-3`, a human calls it "unsafe eval". We measured it: id-based merging left 3 duplicate pairs in the report that location-based merging removed.

</details>

**2. The supervisor caps the report at N findings. What is the minimum it owes the reader when it hits that cap?**

<details><summary>Show answer</summary>

The number it dropped. A cap is a legitimate way to terminate, but silent truncation turns "we found nothing else" and "we stopped looking" into the same output. Our `SynthesisReport` records `dropped_over_cap`, and the run puts it in the trace.

</details>

### Recap

- Limitation we saw: four branches produced 12 findings for 9 defects, and id-based merging left 3 duplicates in the report.
- Layer we added: a bounded fan-in that merges on category plus line proximity, ranks, caps, and reports both `merged_duplicates` and `dropped_over_cap`.
- Evidence it worked: 12 -> 9 findings with 0 duplicates — and a deliberate false merge showing exactly how the same rule can delete a real defect.
